<a href="https://colab.research.google.com/github/aditi-govindu/gemini-recipe-agent/blob/main/agent_loop_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Agent Loop Demo


This notebook demonstrates the core mechanism behind an AI agent: a model that can decide to call a tool, receive the result, and continue reasoning toward a goal — all in a loop.

---

This notebook contains starter code. Please fill in details as needed.
---

## Setup - Gemini API key

How to get the key (takes about two minutes):

* Go to [aistudio.google.com](https://aistudio.google.com/prompts/new_chat)
* Sign in with your Google account
* Click "Get API key" in the left sidebar
* Click "Create API key" — choose "Create API key in new project" if prompted
Copy the key that appears (it starts with AIza...)

> The free tier allows up to 10 requests per minute and 250 requests per day, which is enough to work through this notebook.

If you see occasional slow-downs when you run several questions back to back — that is normal, and the code handles it automatically.

## References
* [Google agent loop](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)
* https://ai.google.dev/gemini-api/docs/thinking#signatures
* https://ai.google.dev/gemini-api/docs/text-generation$0
* [Developer guide](https://github.com/xiaosuhu/midas-ai-in-research/blob/v1.0-dev/docs/notebooks/agent_loop_demo.ipynb$0)



In [4]:
import os

# NEVER share API keys directly. Save them in the environment.
try:
    from google.colab import userdata
    os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')
    print('API key loaded from Colab Secrets.')

# Fall back to manual entry if Secrets are not set up.
# REMOVE API keys before pushing code to Github. This fallback helps.
except Exception:
    import getpass
    key = getpass.getpass('Paste your Gemini API key and press Enter: ')
    os.environ['GOOGLE_API_KEY'] = key
    print('API key set.')

API key loaded from Colab Secrets.


In [5]:
# Quiet ensures your UI is not full of logs.
!pip install google-generativeai --quiet
print("Done.")

Done.


In [7]:
# @title User input needed.

DOCUMENTS = {
    "bharatanatyam": (
        "Bharatanatyam is one of the oldest classical dance forms of India. "
        "It originated in Tamil Nadu. "
        "The dance combines expressions, hand gestures, rhythm and body movements. "
        "It often tells stories from Indian mythology."
    ),

    "western_dance": (
        "Western dance includes several styles such as hip-hop, jazz and contemporary. "
        "Hip-hop developed as part of hip-hop culture. "
        "Contemporary dance combines different movement techniques. "
        "Western dance is performed in competitions, stage shows and entertainment."
    ),

    "dance_choreography": (
        "Choreography is the art of creating and arranging dance movements. "
        "A choreographer plans the movements according to the music. "
        "Choreography includes formations, expressions and transitions. "
        "Dancers practise these movements to create a coordinated performance."
    ),

    "stage_performance": (
        "Stage performances require preparation and regular practice. "
        "Dancers coordinate their movements with music and rhythm. "
        "Costumes and lighting help create the atmosphere of a performance. "
        "Confidence and expressions help dancers connect with the audience."
    )
}

In [9]:
# @title Define a tool

def search_documents(query: str) -> str:
    """Search the document collection for paragraphs relevant to the query."""

    query_lower = query.lower()
    results = []

    for doc_name, content in DOCUMENTS.items():
        words = set(query_lower.split())
        match_count = sum(
            1 for word in words if word in content.lower()
        )

        if match_count > 0:
            results.append((match_count, doc_name, content))

    results.sort(reverse=True)

    if not results:
        return "No relevant documents found for that query."

    # Return the top two matches
    output = []

    for _, name, content in results[:2]:
        output.append(f"[Source: {name}]\n{content}")

    return "\n\n".join(output)


# Sanity check
print(search_documents("classical dance"))

[Source: bharatanatyam]
Bharatanatyam is one of the oldest classical dance forms of India. It originated in Tamil Nadu. The dance combines expressions, hand gestures, rhythm and body movements. It often tells stories from Indian mythology.

[Source: western_dance]
Western dance includes several styles such as hip-hop, jazz and contemporary. Hip-hop developed as part of hip-hop culture. Contemporary dance combines different movement techniques. Western dance is performed in competitions, stage shows and entertainment.


In [10]:
# @title Ask Gemini to search for something

import os
import google.generativeai as genai
import time

genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    tools=[search_documents],
    system_instruction=(
        """
        You are a dance research assistant.
        When you need specific information about dance styles,
        choreography, Bharatanatyam or stage performances,
        use the search_documents tool.
        Only answer based on what you find in the documents.
        If the documents do not contain the answer,
        say so clearly.
        """
    )
)

print("Model ready:", model.model_name)

Model ready: models/gemini-2.5-flash


In [12]:
import time
from google import genai
from google.genai import types

client = genai.Client()

search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="search_documents",
            description="Search the document collection to find the most relevant paragraphs.",
            parameters={
                "type": "object",
                "properties": {
                    "query": {"type": "string"}
                },
                "required": ["query"],
            },
        )
    ]
)

config = types.GenerateContentConfig(
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(
        disable=True
    ),
)

In [24]:
# @title Ask follow up questions

questions = [
    "What are the main features of Bharatanatyam?",
    "How is choreography planned according to music?",
    "What are the important elements of a stage performance?",
    "Which dance styles are included in Western dance?",
    # This question is not answered by the documents.
    "Who invented the telephone?"
]

for i, q in enumerate(questions):
    if i > 0:
        time.sleep(15)  # Stay within the free-tier rate limit.

    print("\n" + "=" * 60);


In [25]:
# This cell will check if you have added atleast 2 more questions.
assert(len(questions) >= 4)
# This will check if you added a question.
assert("Add a question here." not in questions)